# 1. Mocking

## 1.1 What Is Mocking?

Mocking replaces a real dependency with a controllable fake object during a test. This helps you isolate the code you want to test and avoid relying on slow, external, or unpredictable systems.

The main standard-library tool for this in Python is `unittest.mock`.

## 1.2 Basic Tools

The most common tools are:

- `Mock`: a general-purpose fake object
- `MagicMock`: like `Mock`, but with built-in support for magic methods
- `patch`: temporarily replaces an object during a test

## 1.3 Why Use Mocking

Mocking is useful because it improves:

- speed, by avoiding real network calls or databases
- determinism, by making dependencies return predictable values
- isolation, by keeping tests focused on one unit of code
- simplicity, by replacing complex collaborators with smaller fakes

## 2. Simple Mock Example

In [1]:
from unittest.mock import Mock

# Create a mock object
mock_obj = Mock()

# Set a return value
mock_obj.method.return_value = 42
print(mock_obj.method())  # Output: 42

# Set side effects
mock_obj.method_with_side_effect = Mock(side_effect=[1, 2, 3])
print(mock_obj.method_with_side_effect())  # Output: 1
print(mock_obj.method_with_side_effect())  # Output: 2
print(mock_obj.method_with_side_effect())  # Output: 3

# Raise an exception
mock_obj.error_method = Mock(side_effect=ValueError("Something went wrong"))
try:
    mock_obj.error_method()
except ValueError as e:
    print(f"Caught exception: {e}")

42
1
2
3
Caught exception: Something went wrong


## 2.1 When to Use `Mock`

Use `Mock` when your code already accepts the dependency as an argument or attribute, so you can inject a fake object directly in the test.

In [ ]:
import unittest
from unittest.mock import Mock

# Code under test
def send_welcome(user, email_client):
    email_client.send(user.email, "Welcome!")
    return True

# Test case
class TestSendWelcome(unittest.TestCase):
    def test_send_welcome(self):
        #  You create a fake object manually in your test and pass it into your code.
        fake_email_client = Mock()
        user = Mock()
        user.email = "user@example.com"

        result = send_welcome(user, fake_email_client)

        # Assertions
        self.assertTrue(result)
        fake_email_client.send.assert_called_once_with("user@example.com", "Welcome!")

# Entry point
if __name__ == "__main__":
    unittest.main(argv=['first-arg-is-ignored'], exit=False)


.
----------------------------------------------------------------------
Ran 1 test in 0.002s

OK


## 2.2 When to Use `MagicMock`

`MagicMock` is helpful when the object being replaced needs magic methods such as `__len__`, `__iter__`, `__getitem__`, or context-manager methods like `__enter__` and `__exit__`.

In [ ]:
# Example: Mocking a file object
from unittest.mock import MagicMock

mock_file = MagicMock()
mock_file.__enter__.return_value.read.return_value = "Hello World"

with mock_file as f:
    data = f.read()
print(data)  # Hello World


Hello World


In [ ]:
# Mocking Iterable
mock_list = MagicMock()
mock_list.__iter__.return_value = iter([1, 2, 3])

for item in mock_list:
    print(item)


1
2
3


## 2.3 When to Use `patch`

Use `patch` when the dependency is created or imported inside the code under test and you cannot conveniently inject it yourself. In that case, patch the name used by the module under test.

In [ ]:
# my_module.py
import requests

def fetch_data():
    response = requests.get("https://api.com/data")
    return response.json()


In [ ]:
import unittest
from unittest.mock import patch

class TestSendWelcome(unittest.TestCase):
    @patch("my_module.requests.get")  # patch the correct import path
    def test_fetch_data(self, mock_get):
        mock_get.return_value.json.return_value = {"ok": True}

        from my_module import fetch_data
        result = fetch_data()

        self.assertEqual(result, {"ok": True})
        mock_get.assert_called_once_with("https://api.com/data")

if __name__ == "__main__":
    unittest.main()

In [ ]:
from unittest.mock import patch

# Function we want to test
def get_user_email(user_id):
    # This would normally call a database or API
    return f"user{user_id}@example.com"

# Test with patch
with patch('__main__.get_user_email') as mock_get_email:
    mock_get_email.return_value = 'test@example.com'

    # During this context, get_user_email is replaced with our mock
    result = get_user_email(1)
    print(result)  # Output: test@example.com

# Outside the context, the original function is restored
print(get_user_email(1))  # Output: user1@example.com

test@example.com
user1@example.com


## Key Takeaway: Choosing the Right Mocking Tool

- **Mock** → Use when you can inject the dependency directly.  
- **MagicMock** → Use when the dependency uses magic/dunder methods (context managers, iterables, indexing, etc.).  
- **patch** → Use when the dependency is imported or created inside the module under test.


## 3. References

Useful references for deeper study:

1. Official documentation for `unittest.mock`: https://docs.python.org/3/library/unittest.mock.html
2. The Python testing guide: https://docs.python.org/3/library/unittest.html